{
 "cells": [
  {
   "metadata": {
    "ExecuteTime": {
     "end_time": "2026-03-27T12:54:46.758511Z",
     "start_time": "2026-03-27T12:54:42.021555Z"
    }
   },
   "cell_type": "code",
   "source": [
    "import torch\n",
    "import torch.nn.functional as F\n",
    "import torch.nn as nn\n",
    "from torchvision import datasets, transforms\n",
    "from torch.utils.data import DataLoader\n",
    "import matplotlib.pyplot as plt\n",
    "from definitions import ROOT_DIR, MPW_CNN_DIR\n",
    "import src.cnn_utils"
   ],
   "id": "1d57edf58cdfd15f",
   "outputs": [],
   "execution_count": 2
  },
  {
   "metadata": {},
   "cell_type": "markdown",
   "source": "# Shallow SmallCNN",
   "id": "a75b40a208225aa3"
  },
  {
   "metadata": {
    "ExecuteTime": {
     "end_time": "2026-03-27T12:54:46.764764Z",
     "start_time": "2026-03-27T12:54:46.759388Z"
    }
   },
   "cell_type": "code",
   "source": [
    "class SmallCNN(nn.Module):\n",
    "    def __init__(self, num_classes=10):\n",
    "        super().__init__()\n",
    "        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)\n",
    "        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)\n",
    "        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)\n",
    "\n",
    "        self.pool = nn.MaxPool2d(2, 2)\n",
    "        self.gap = nn.AdaptiveAvgPool2d((1, 1))\n",
    "        self.fc = nn.Linear(64, num_classes)\n",
    "\n",
    "    def forward(self, x):\n",
    "        x = self.pool(F.relu(self.conv1(x)))\n",
    "        x = self.pool(F.relu(self.conv2(x)))\n",
    "        x = F.relu(self.conv3(x))      # last conv layer -> Grad-CAM target\n",
    "        x = self.gap(x)\n",
    "        x = torch.flatten(x, 1)\n",
    "        x = self.fc(x)                 # raw logits\n",
    "        return x"
   ],
   "id": "fc22ec95dccbbad7",
   "outputs": [],
   "execution_count": 3
  },
  {
   "metadata": {},
   "cell_type": "markdown",
   "source": "# Transforms and loaders\n",
   "id": "ecf56e9f2e312958"
  },
  {
   "metadata": {
    "ExecuteTime": {
     "end_time": "2026-03-27T12:54:46.947533Z",
     "start_time": "2026-03-27T12:54:46.901690Z"
    }
   },
   "cell_type": "code",
   "source": [
    "from torchvision import datasets, transforms\n",
    "from torch.utils.data import DataLoader\n",
    "\n",
    "img_size = 224\n",
    "batch_size = 32\n",
    "\n",
    "transformer = transforms.Compose([\n",
    "    transforms.Resize((img_size, img_size)),\n",
    "    transforms.ToTensor(),\n",
    "    transforms.Normalize(mean=[0.5, 0.5, 0.5],\n",
    "                         std=[0.5, 0.5, 0.5]),\n",
    "])\n",
    "\n",
    "full_dataset = datasets.ImageFolder(\"../data/icosimal_img_class_03/train\", transform=transformer)\n",
    "splits = torch.load(\"../data/split/split_train_test_indices.pth\")\n",
    "\n",
    "train_ds = torch.utils.data.Subset(full_dataset, splits['train_idx'])\n",
    "test_ds = torch.utils.data.Subset(full_dataset, splits['test_idx'])\n",
    "val_ds = datasets.ImageFolder(\"../data/icosimal_img_class_03/validate\", transform=transformer)\n",
    "\n",
    "train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)\n",
    "val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)\n",
    "test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)\n",
    "\n",
    "print(f\"Training dataset size: {len(train_ds)}\")\n",
    "print(f\"Validation dataset size: {len(val_ds)}\")\n",
    "print(f\"Test dataset size: {len(test_ds)}\")\n",
    "\n",
    "class_names = val_ds.classes\n",
    "num_classes = len(class_names)\n",
    "class_to_idx = val_ds.class_to_idx\n",
    "print(\"Number of classes: \", num_classes)\n",
    "print(\"Class names: \", class_names)"
   ],
   "id": "222e5f2a729e7920",
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Training dataset size: 20000\n",
      "Validation dataset size: 6000\n",
      "Test dataset size: 4000\n",
      "Number of classes:  10\n",
      "Class names:  ['cat', 'chicken', 'cow', 'dog', 'elephant', 'horse', 'rabbit', 'sheep', 'squirrel', 'zebra']\n"
     ]
    }
   ],
   "execution_count": 4
  },
  {
   "metadata": {},
   "cell_type": "markdown",
   "source": "# Device",
   "id": "8922a373da28b3f8"
  },
  {
   "metadata": {
    "ExecuteTime": {
     "end_time": "2026-03-27T12:54:51.745875Z",
     "start_time": "2026-03-27T12:54:51.680842Z"
    }
   },
   "cell_type": "code",
   "source": [
    "# Check for GPU\n",
    "device = None\n",
    "if torch.cuda.is_available():\n",
    "    device = torch.device(\"cuda\")\n",
    "elif torch.backends.mps.is_available():\n",
    "    device = torch.device(\"mps\")\n",
    "else:\n",
    "    device = torch.device(\"cpu\")\n",
    "\n",
    "print(\"Current device : \", device)"
   ],
   "id": "9d8a4f49869ad3a9",
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Current device :  mps\n"
     ]
    }
   ],
   "execution_count": 5
  },
  {
   "metadata": {},
   "cell_type": "markdown",
   "source": "# Model, loss, optimizer",
   "id": "f78cb6f12d9bf1f"
  },
  {
   "metadata": {
    "ExecuteTime": {
     "end_time": "2026-03-27T12:54:54.343733Z",
     "start_time": "2026-03-27T12:54:54.325341Z"
    }
   },
   "cell_type": "code",
   "source": [
    "model = SmallCNN(num_classes=num_classes).to(device)\n",
    "criterion = nn.CrossEntropyLoss()\n",
    "optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)"
   ],
   "id": "4957cd0576728228",
   "outputs": [],
   "execution_count": 6
  },
  {
   "metadata": {},
   "cell_type": "markdown",
   "source": "# Train",
   "id": "4d86fbde3b3dd5b9"
  },
  {
   "metadata": {
    "ExecuteTime": {
     "end_time": "2026-03-27T12:54:58.512126Z",
     "start_time": "2026-03-27T12:54:58.494047Z"
    }
   },
   "cell_type": "code",
   "source": [
    "def train_one_epoch(model, loader, criterion, optimizer, device):\n",
    "    model.train()\n",
    "    running_loss = 0.0\n",
    "    correct = 0\n",
    "    total = 0\n",
    "\n",
    "    for images, labels in loader:\n",
    "        images = images.to(device)\n",
    "        labels = labels.to(device)\n",
    "\n",
    "        optimizer.zero_grad()\n",
    "        logits = model(images)\n",
    "        loss = criterion(logits, labels)\n",
    "        loss.backward()\n",
    "        optimizer.step()\n",
    "\n",
    "        running_loss += loss.item() * images.size(0)\n",
    "        preds = logits.argmax(dim=1)\n",
    "        correct += (preds == labels).sum().item()\n",
    "        total += labels.size(0)\n",
    "\n",
    "    epoch_loss = running_loss / total\n",
    "    epoch_acc = correct / total\n",
    "    return epoch_loss, epoch_acc"
   ],
   "id": "377b9fc8e394dc39",
   "outputs": [],
   "execution_count": 7
  },
  {
   "metadata": {
    "ExecuteTime": {
     "end_time": "2026-03-27T12:55:00.459409Z",
     "start_time": "2026-03-27T12:55:00.446713Z"
    }
   },
   "cell_type": "code",
   "source": [
    "@torch.no_grad()\n",
    "def evaluate(model, loader, criterion, device):\n",
    "    model.eval()\n",
    "    running_loss = 0.0\n",
    "    correct = 0\n",
    "    total = 0\n",
    "\n",
    "    for images, labels in loader:\n",
    "        images = images.to(device)\n",
    "        labels = labels.to(device)\n",
    "\n",
    "        logits = model(images)\n",
    "        loss = criterion(logits, labels)\n",
    "\n",
    "        running_loss += loss.item() * images.size(0)\n",
    "        preds = logits.argmax(dim=1)\n",
    "        correct += (preds == labels).sum().item()\n",
    "        total += labels.size(0)\n",
    "\n",
    "    epoch_loss = running_loss / total\n",
    "    epoch_acc = correct / total\n",
    "    return epoch_loss, epoch_acc"
   ],
   "id": "9f48ec0f03fb6aa9",
   "outputs": [],
   "execution_count": 8
  },
  {
   "metadata": {},
   "cell_type": "markdown",
   "source": "# Loop",
   "id": "784b59b534c25f65"
  },
  {
   "metadata": {
    "ExecuteTime": {
     "end_time": "2026-03-25T12:08:04.813017Z",
     "start_time": "2026-03-25T12:08:00.300098Z"
    }
   },
   "cell_type": "code",
   "source": [
    "num_epochs = 10\n",
    "best_val_acc = 0.0\n",
    "\n",
    "for epoch in range(num_epochs):\n",
    "    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)\n",
    "    val_loss, val_acc = evaluate(model, val_loader, criterion, device)\n",
    "\n",
    "    print(\n",
    "        f\"Epoch {epoch+1:02d}/{num_epochs} | \"\n",
    "        f\"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | \"\n",
    "        f\"val_loss={val_loss:.4f} val_acc={val_acc:.4f}\"\n",
    "    )\n",
    "\n",
    "    if val_acc > best_val_acc:\n",
    "        best_val_acc = val_acc\n",
    "        torch.save(model.state_dict(), \"best_smallcnn.pt\")\n",
    "\n",
    "print(\"Best val acc:\", best_val_acc)"
   ],
   "id": "ab6d8367bd830253",
   "outputs": [
    {
     "ename": "KeyboardInterrupt",
     "evalue": "",
     "output_type": "error",
     "traceback": [
      "\u001B[31m---------------------------------------------------------------------------\u001B[39m",
      "\u001B[31mKeyboardInterrupt\u001B[39m                         Traceback (most recent call last)",
      "\u001B[36mCell\u001B[39m\u001B[36m \u001B[39m\u001B[32mIn[7]\u001B[39m\u001B[32m, line 5\u001B[39m\n\u001B[32m      2\u001B[39m best_val_acc = \u001B[32m0.0\u001B[39m\n\u001B[32m      4\u001B[39m \u001B[38;5;28;01mfor\u001B[39;00m epoch \u001B[38;5;129;01min\u001B[39;00m \u001B[38;5;28mrange\u001B[39m(num_epochs):\n\u001B[32m----> \u001B[39m\u001B[32m5\u001B[39m     train_loss, train_acc = \u001B[43mtrain_one_epoch\u001B[49m\u001B[43m(\u001B[49m\u001B[43mmodel\u001B[49m\u001B[43m,\u001B[49m\u001B[43m \u001B[49m\u001B[43mtrain_loader\u001B[49m\u001B[43m,\u001B[49m\u001B[43m \u001B[49m\u001B[43mcriterion\u001B[49m\u001B[43m,\u001B[49m\u001B[43m \u001B[49m\u001B[43moptimizer\u001B[49m\u001B[43m,\u001B[49m\u001B[43m \u001B[49m\u001B[43mdevice\u001B[49m\u001B[43m)\u001B[49m\n\u001B[32m      6\u001B[39m     val_loss, val_acc = evaluate(model, val_loader, criterion, device)\n\u001B[32m      8\u001B[39m     \u001B[38;5;28mprint\u001B[39m(\n\u001B[32m      9\u001B[39m         \u001B[33mf\u001B[39m\u001B[33m\"\u001B[39m\u001B[33mEpoch \u001B[39m\u001B[38;5;132;01m{\u001B[39;00mepoch+\u001B[32m1\u001B[39m\u001B[38;5;132;01m:\u001B[39;00m\u001B[33m02d\u001B[39m\u001B[38;5;132;01m}\u001B[39;00m\u001B[33m/\u001B[39m\u001B[38;5;132;01m{\u001B[39;00mnum_epochs\u001B[38;5;132;01m}\u001B[39;00m\u001B[33m | \u001B[39m\u001B[33m\"\u001B[39m\n\u001B[32m     10\u001B[39m         \u001B[33mf\u001B[39m\u001B[33m\"\u001B[39m\u001B[33mtrain_loss=\u001B[39m\u001B[38;5;132;01m{\u001B[39;00mtrain_loss\u001B[38;5;132;01m:\u001B[39;00m\u001B[33m.4f\u001B[39m\u001B[38;5;132;01m}\u001B[39;00m\u001B[33m train_acc=\u001B[39m\u001B[38;5;132;01m{\u001B[39;00mtrain_acc\u001B[38;5;132;01m:\u001B[39;00m\u001B[33m.4f\u001B[39m\u001B[38;5;132;01m}\u001B[39;00m\u001B[33m | \u001B[39m\u001B[33m\"\u001B[39m\n\u001B[32m     11\u001B[39m         \u001B[33mf\u001B[39m\u001B[33m\"\u001B[39m\u001B[33mval_loss=\u001B[39m\u001B[38;5;132;01m{\u001B[39;00mval_loss\u001B[38;5;132;01m:\u001B[39;00m\u001B[33m.4f\u001B[39m\u001B[38;5;132;01m}\u001B[39;00m\u001B[33m val_acc=\u001B[39m\u001B[38;5;132;01m{\u001B[39;00mval_acc\u001B[38;5;132;01m:\u001B[39;00m\u001B[33m.4f\u001B[39m\u001B[38;5;132;01m}\u001B[39;00m\u001B[33m\"\u001B[39m\n\u001B[32m     12\u001B[39m     )\n",
      "\u001B[36mCell\u001B[39m\u001B[36m \u001B[39m\u001B[32mIn[6]\u001B[39m\u001B[32m, line 7\u001B[39m, in \u001B[36mtrain_one_epoch\u001B[39m\u001B[34m(model, loader, criterion, optimizer, device)\u001B[39m\n\u001B[32m      4\u001B[39m correct = \u001B[32m0\u001B[39m\n\u001B[32m      5\u001B[39m total = \u001B[32m0\u001B[39m\n\u001B[32m----> \u001B[39m\u001B[32m7\u001B[39m \u001B[43m\u001B[49m\u001B[38;5;28;43;01mfor\u001B[39;49;00m\u001B[43m \u001B[49m\u001B[43mimages\u001B[49m\u001B[43m,\u001B[49m\u001B[43m \u001B[49m\u001B[43mlabels\u001B[49m\u001B[43m \u001B[49m\u001B[38;5;129;43;01min\u001B[39;49;00m\u001B[43m \u001B[49m\u001B[43mloader\u001B[49m\u001B[43m:\u001B[49m\n\u001B[32m      8\u001B[39m \u001B[43m    \u001B[49m\u001B[43mimages\u001B[49m\u001B[43m \u001B[49m\u001B[43m=\u001B[49m\u001B[43m \u001B[49m\u001B[43mimages\u001B[49m\u001B[43m.\u001B[49m\u001B[43mto\u001B[49m\u001B[43m(\u001B[49m\u001B[43mdevice\u001B[49m\u001B[43m)\u001B[49m\n\u001B[32m      9\u001B[39m \u001B[43m    \u001B[49m\u001B[43mlabels\u001B[49m\u001B[43m \u001B[49m\u001B[43m=\u001B[49m\u001B[43m \u001B[49m\u001B[43mlabels\u001B[49m\u001B[43m.\u001B[49m\u001B[43mto\u001B[49m\u001B[43m(\u001B[49m\u001B[43mdevice\u001B[49m\u001B[43m)\u001B[49m\n",
      "\u001B[36mFile \u001B[39m\u001B[32m~/Projects/PycharmProjects/deep-learning-mpw/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:741\u001B[39m, in \u001B[36m_BaseDataLoaderIter.__next__\u001B[39m\u001B[34m(self)\u001B[39m\n\u001B[32m    738\u001B[39m \u001B[38;5;28;01mif\u001B[39;00m \u001B[38;5;28mself\u001B[39m._sampler_iter \u001B[38;5;129;01mis\u001B[39;00m \u001B[38;5;28;01mNone\u001B[39;00m:\n\u001B[32m    739\u001B[39m     \u001B[38;5;66;03m# TODO(https://github.com/pytorch/pytorch/issues/76750)\u001B[39;00m\n\u001B[32m    740\u001B[39m     \u001B[38;5;28mself\u001B[39m._reset()  \u001B[38;5;66;03m# type: ignore[call-arg]\u001B[39;00m\n\u001B[32m--> \u001B[39m\u001B[32m741\u001B[39m data = \u001B[38;5;28;43mself\u001B[39;49m\u001B[43m.\u001B[49m\u001B[43m_next_data\u001B[49m\u001B[43m(\u001B[49m\u001B[43m)\u001B[49m\n\u001B[32m    742\u001B[39m \u001B[38;5;28mself\u001B[39m._num_yielded += \u001B[32m1\u001B[39m\n\u001B[32m    743\u001B[39m \u001B[38;5;28;01mif\u001B[39;00m (\n\u001B[32m    744\u001B[39m     \u001B[38;5;28mself\u001B[39m._dataset_kind == _DatasetKind.Iterable\n\u001B[32m    745\u001B[39m     \u001B[38;5;129;01mand\u001B[39;00m \u001B[38;5;28mself\u001B[39m._IterableDataset_len_called \u001B[38;5;129;01mis\u001B[39;00m \u001B[38;5;129;01mnot\u001B[39;00m \u001B[38;5;28;01mNone\u001B[39;00m\n\u001B[32m    746\u001B[39m     \u001B[38;5;129;01mand\u001B[39;00m \u001B[38;5;28mself\u001B[39m._num_yielded > \u001B[38;5;28mself\u001B[39m._IterableDataset_len_called\n\u001B[32m    747\u001B[39m ):\n",
      "\u001B[36mFile \u001B[39m\u001B[32m~/Projects/PycharmProjects/deep-learning-mpw/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:801\u001B[39m, in \u001B[36m_SingleProcessDataLoaderIter._next_data\u001B[39m\u001B[34m(self)\u001B[39m\n\u001B[32m    799\u001B[39m \u001B[38;5;28;01mdef\u001B[39;00m\u001B[38;5;250m \u001B[39m\u001B[34m_next_data\u001B[39m(\u001B[38;5;28mself\u001B[39m):\n\u001B[32m    800\u001B[39m     index = \u001B[38;5;28mself\u001B[39m._next_index()  \u001B[38;5;66;03m# may raise StopIteration\u001B[39;00m\n\u001B[32m--> \u001B[39m\u001B[32m801\u001B[39m     data = \u001B[38;5;28;43mself\u001B[39;49m\u001B[43m.\u001B[49m\u001B[43m_dataset_fetcher\u001B[49m\u001B[43m.\u001B[49m\u001B[43mfetch\u001B[49m\u001B[43m(\u001B[49m\u001B[43mindex\u001B[49m\u001B[43m)\u001B[49m  \u001B[38;5;66;03m# may raise StopIteration\u001B[39;00m\n\u001B[32m    802\u001B[39m     \u001B[38;5;28;01mif\u001B[39;00m \u001B[38;5;28mself\u001B[39m._pin_memory:\n\u001B[32m    803\u001B[39m         data = _utils.pin_memory.pin_memory(data, \u001B[38;5;28mself\u001B[39m._pin_memory_device)\n",
      "\u001B[36mFile \u001B[39m\u001B[32m~/Projects/PycharmProjects/deep-learning-mpw/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/fetch.py:52\u001B[39m, in \u001B[36m_MapDatasetFetcher.fetch\u001B[39m\u001B[34m(self, possibly_batched_index)\u001B[39m\n\u001B[32m     50\u001B[39m \u001B[38;5;28;01mif\u001B[39;00m \u001B[38;5;28mself\u001B[39m.auto_collation:\n\u001B[32m     51\u001B[39m     \u001B[38;5;28;01mif\u001B[39;00m \u001B[38;5;28mhasattr\u001B[39m(\u001B[38;5;28mself\u001B[39m.dataset, \u001B[33m\"\u001B[39m\u001B[33m__getitems__\u001B[39m\u001B[33m\"\u001B[39m) \u001B[38;5;129;01mand\u001B[39;00m \u001B[38;5;28mself\u001B[39m.dataset.__getitems__:\n\u001B[32m---> \u001B[39m\u001B[32m52\u001B[39m         data = \u001B[38;5;28;43mself\u001B[39;49m\u001B[43m.\u001B[49m\u001B[43mdataset\u001B[49m\u001B[43m.\u001B[49m\u001B[43m__getitems__\u001B[49m\u001B[43m(\u001B[49m\u001B[43mpossibly_batched_index\u001B[49m\u001B[43m)\u001B[49m\n\u001B[32m     53\u001B[39m     \u001B[38;5;28;01melse\u001B[39;00m:\n\u001B[32m     54\u001B[39m         data = [\u001B[38;5;28mself\u001B[39m.dataset[idx] \u001B[38;5;28;01mfor\u001B[39;00m idx \u001B[38;5;129;01min\u001B[39;00m possibly_batched_index]\n",
      "\u001B[36mFile \u001B[39m\u001B[32m~/Projects/PycharmProjects/deep-learning-mpw/.venv/lib/python3.12/site-packages/torch/utils/data/dataset.py:413\u001B[39m, in \u001B[36mSubset.__getitems__\u001B[39m\u001B[34m(self, indices)\u001B[39m\n\u001B[32m    411\u001B[39m     \u001B[38;5;28;01mreturn\u001B[39;00m \u001B[38;5;28mself\u001B[39m.dataset.__getitems__([\u001B[38;5;28mself\u001B[39m.indices[idx] \u001B[38;5;28;01mfor\u001B[39;00m idx \u001B[38;5;129;01min\u001B[39;00m indices])  \u001B[38;5;66;03m# type: ignore[attr-defined]\u001B[39;00m\n\u001B[32m    412\u001B[39m \u001B[38;5;28;01melse\u001B[39;00m:\n\u001B[32m--> \u001B[39m\u001B[32m413\u001B[39m     \u001B[38;5;28;01mreturn\u001B[39;00m [\u001B[38;5;28;43mself\u001B[39;49m\u001B[43m.\u001B[49m\u001B[43mdataset\u001B[49m\u001B[43m[\u001B[49m\u001B[38;5;28;43mself\u001B[39;49m\u001B[43m.\u001B[49m\u001B[43mindices\u001B[49m\u001B[43m[\u001B[49m\u001B[43midx\u001B[49m\u001B[43m]\u001B[49m\u001B[43m]\u001B[49m \u001B[38;5;28;01mfor\u001B[39;00m idx \u001B[38;5;129;01min\u001B[39;00m indices]\n",
      "\u001B[36mFile \u001B[39m\u001B[32m~/Projects/PycharmProjects/deep-learning-mpw/.venv/lib/python3.12/site-packages/torchvision/datasets/folder.py:247\u001B[39m, in \u001B[36mDatasetFolder.__getitem__\u001B[39m\u001B[34m(self, index)\u001B[39m\n\u001B[32m    245\u001B[39m sample = \u001B[38;5;28mself\u001B[39m.loader(path)\n\u001B[32m    246\u001B[39m \u001B[38;5;28;01mif\u001B[39;00m \u001B[38;5;28mself\u001B[39m.transform \u001B[38;5;129;01mis\u001B[39;00m \u001B[38;5;129;01mnot\u001B[39;00m \u001B[38;5;28;01mNone\u001B[39;00m:\n\u001B[32m--> \u001B[39m\u001B[32m247\u001B[39m     sample = \u001B[38;5;28;43mself\u001B[39;49m\u001B[43m.\u001B[49m\u001B[43mtransform\u001B[49m\u001B[43m(\u001B[49m\u001B[43msample\u001B[49m\u001B[43m)\u001B[49m\n\u001B[32m    248\u001B[39m \u001B[38;5;28;01mif\u001B[39;00m \u001B[38;5;28mself\u001B[39m.target_transform \u001B[38;5;129;01mis\u001B[39;00m \u001B[38;5;129;01mnot\u001B[39;00m \u001B[38;5;28;01mNone\u001B[39;00m:\n\u001B[32m    249\u001B[39m     target = \u001B[38;5;28mself\u001B[39m.target_transform(target)\n",
      "\u001B[36mFile \u001B[39m\u001B[32m~/Projects/PycharmProjects/deep-learning-mpw/.venv/lib/python3.12/site-packages/torchvision/transforms/transforms.py:95\u001B[39m, in \u001B[36mCompose.__call__\u001B[39m\u001B[34m(self, img)\u001B[39m\n\u001B[32m     93\u001B[39m \u001B[38;5;28;01mdef\u001B[39;00m\u001B[38;5;250m \u001B[39m\u001B[34m__call__\u001B[39m(\u001B[38;5;28mself\u001B[39m, img):\n\u001B[32m     94\u001B[39m     \u001B[38;5;28;01mfor\u001B[39;00m t \u001B[38;5;129;01min\u001B[39;00m \u001B[38;5;28mself\u001B[39m.transforms:\n\u001B[32m---> \u001B[39m\u001B[32m95\u001B[39m         img = \u001B[43mt\u001B[49m\u001B[43m(\u001B[49m\u001B[43mimg\u001B[49m\u001B[43m)\u001B[49m\n\u001B[32m     96\u001B[39m     \u001B[38;5;28;01mreturn\u001B[39;00m img\n",
      "\u001B[36mFile \u001B[39m\u001B[32m~/Projects/PycharmProjects/deep-learning-mpw/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1776\u001B[39m, in \u001B[36mModule._wrapped_call_impl\u001B[39m\u001B[34m(self, *args, **kwargs)\u001B[39m\n\u001B[32m   1774\u001B[39m     \u001B[38;5;28;01mreturn\u001B[39;00m \u001B[38;5;28mself\u001B[39m._compiled_call_impl(*args, **kwargs)  \u001B[38;5;66;03m# type: ignore[misc]\u001B[39;00m\n\u001B[32m   1775\u001B[39m \u001B[38;5;28;01melse\u001B[39;00m:\n\u001B[32m-> \u001B[39m\u001B[32m1776\u001B[39m     \u001B[38;5;28;01mreturn\u001B[39;00m \u001B[38;5;28;43mself\u001B[39;49m\u001B[43m.\u001B[49m\u001B[43m_call_impl\u001B[49m\u001B[43m(\u001B[49m\u001B[43m*\u001B[49m\u001B[43margs\u001B[49m\u001B[43m,\u001B[49m\u001B[43m \u001B[49m\u001B[43m*\u001B[49m\u001B[43m*\u001B[49m\u001B[43mkwargs\u001B[49m\u001B[43m)\u001B[49m\n",
      "\u001B[36mFile \u001B[39m\u001B[32m~/Projects/PycharmProjects/deep-learning-mpw/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1787\u001B[39m, in \u001B[36mModule._call_impl\u001B[39m\u001B[34m(self, *args, **kwargs)\u001B[39m\n\u001B[32m   1782\u001B[39m \u001B[38;5;66;03m# If we don't have any hooks, we want to skip the rest of the logic in\u001B[39;00m\n\u001B[32m   1783\u001B[39m \u001B[38;5;66;03m# this function, and just call forward.\u001B[39;00m\n\u001B[32m   1784\u001B[39m \u001B[38;5;28;01mif\u001B[39;00m \u001B[38;5;129;01mnot\u001B[39;00m (\u001B[38;5;28mself\u001B[39m._backward_hooks \u001B[38;5;129;01mor\u001B[39;00m \u001B[38;5;28mself\u001B[39m._backward_pre_hooks \u001B[38;5;129;01mor\u001B[39;00m \u001B[38;5;28mself\u001B[39m._forward_hooks \u001B[38;5;129;01mor\u001B[39;00m \u001B[38;5;28mself\u001B[39m._forward_pre_hooks\n\u001B[32m   1785\u001B[39m         \u001B[38;5;129;01mor\u001B[39;00m _global_backward_pre_hooks \u001B[38;5;129;01mor\u001B[39;00m _global_backward_hooks\n\u001B[32m   1786\u001B[39m         \u001B[38;5;129;01mor\u001B[39;00m _global_forward_hooks \u001B[38;5;129;01mor\u001B[39;00m _global_forward_pre_hooks):\n\u001B[32m-> \u001B[39m\u001B[32m1787\u001B[39m     \u001B[38;5;28;01mreturn\u001B[39;00m \u001B[43mforward_call\u001B[49m\u001B[43m(\u001B[49m\u001B[43m*\u001B[49m\u001B[43margs\u001B[49m\u001B[43m,\u001B[49m\u001B[43m \u001B[49m\u001B[43m*\u001B[49m\u001B[43m*\u001B[49m\u001B[43mkwargs\u001B[49m\u001B[43m)\u001B[49m\n\u001B[32m   1789\u001B[39m result = \u001B[38;5;28;01mNone\u001B[39;00m\n\u001B[32m   1790\u001B[39m called_always_called_hooks = \u001B[38;5;28mset\u001B[39m()\n",
      "\u001B[36mFile \u001B[39m\u001B[32m~/Projects/PycharmProjects/deep-learning-mpw/.venv/lib/python3.12/site-packages/torchvision/transforms/transforms.py:285\u001B[39m, in \u001B[36mNormalize.forward\u001B[39m\u001B[34m(self, tensor)\u001B[39m\n\u001B[32m    277\u001B[39m \u001B[38;5;28;01mdef\u001B[39;00m\u001B[38;5;250m \u001B[39m\u001B[34mforward\u001B[39m(\u001B[38;5;28mself\u001B[39m, tensor: Tensor) -> Tensor:\n\u001B[32m    278\u001B[39m \u001B[38;5;250m    \u001B[39m\u001B[33;03m\"\"\"\u001B[39;00m\n\u001B[32m    279\u001B[39m \u001B[33;03m    Args:\u001B[39;00m\n\u001B[32m    280\u001B[39m \u001B[33;03m        tensor (Tensor): Tensor image to be normalized.\u001B[39;00m\n\u001B[32m   (...)\u001B[39m\u001B[32m    283\u001B[39m \u001B[33;03m        Tensor: Normalized Tensor image.\u001B[39;00m\n\u001B[32m    284\u001B[39m \u001B[33;03m    \"\"\"\u001B[39;00m\n\u001B[32m--> \u001B[39m\u001B[32m285\u001B[39m     \u001B[38;5;28;01mreturn\u001B[39;00m \u001B[43mF\u001B[49m\u001B[43m.\u001B[49m\u001B[43mnormalize\u001B[49m\u001B[43m(\u001B[49m\u001B[43mtensor\u001B[49m\u001B[43m,\u001B[49m\u001B[43m \u001B[49m\u001B[38;5;28;43mself\u001B[39;49m\u001B[43m.\u001B[49m\u001B[43mmean\u001B[49m\u001B[43m,\u001B[49m\u001B[43m \u001B[49m\u001B[38;5;28;43mself\u001B[39;49m\u001B[43m.\u001B[49m\u001B[43mstd\u001B[49m\u001B[43m,\u001B[49m\u001B[43m \u001B[49m\u001B[38;5;28;43mself\u001B[39;49m\u001B[43m.\u001B[49m\u001B[43minplace\u001B[49m\u001B[43m)\u001B[49m\n",
      "\u001B[36mFile \u001B[39m\u001B[32m~/Projects/PycharmProjects/deep-learning-mpw/.venv/lib/python3.12/site-packages/torchvision/transforms/functional.py:350\u001B[39m, in \u001B[36mnormalize\u001B[39m\u001B[34m(tensor, mean, std, inplace)\u001B[39m\n\u001B[32m    347\u001B[39m \u001B[38;5;28;01mif\u001B[39;00m \u001B[38;5;129;01mnot\u001B[39;00m \u001B[38;5;28misinstance\u001B[39m(tensor, torch.Tensor):\n\u001B[32m    348\u001B[39m     \u001B[38;5;28;01mraise\u001B[39;00m \u001B[38;5;167;01mTypeError\u001B[39;00m(\u001B[33mf\u001B[39m\u001B[33m\"\u001B[39m\u001B[33mimg should be Tensor Image. Got \u001B[39m\u001B[38;5;132;01m{\u001B[39;00m\u001B[38;5;28mtype\u001B[39m(tensor)\u001B[38;5;132;01m}\u001B[39;00m\u001B[33m\"\u001B[39m)\n\u001B[32m--> \u001B[39m\u001B[32m350\u001B[39m \u001B[38;5;28;01mreturn\u001B[39;00m \u001B[43mF_t\u001B[49m\u001B[43m.\u001B[49m\u001B[43mnormalize\u001B[49m\u001B[43m(\u001B[49m\u001B[43mtensor\u001B[49m\u001B[43m,\u001B[49m\u001B[43m \u001B[49m\u001B[43mmean\u001B[49m\u001B[43m=\u001B[49m\u001B[43mmean\u001B[49m\u001B[43m,\u001B[49m\u001B[43m \u001B[49m\u001B[43mstd\u001B[49m\u001B[43m=\u001B[49m\u001B[43mstd\u001B[49m\u001B[43m,\u001B[49m\u001B[43m \u001B[49m\u001B[43minplace\u001B[49m\u001B[43m=\u001B[49m\u001B[43minplace\u001B[49m\u001B[43m)\u001B[49m\n",
      "\u001B[36mFile \u001B[39m\u001B[32m~/Projects/PycharmProjects/deep-learning-mpw/.venv/lib/python3.12/site-packages/torchvision/transforms/_functional_tensor.py:917\u001B[39m, in \u001B[36mnormalize\u001B[39m\u001B[34m(tensor, mean, std, inplace)\u001B[39m\n\u001B[32m    912\u001B[39m     \u001B[38;5;28;01mraise\u001B[39;00m \u001B[38;5;167;01mValueError\u001B[39;00m(\n\u001B[32m    913\u001B[39m         \u001B[33mf\u001B[39m\u001B[33m\"\u001B[39m\u001B[33mExpected tensor to be a tensor image of size (..., C, H, W). Got tensor.size() = \u001B[39m\u001B[38;5;132;01m{\u001B[39;00mtensor.size()\u001B[38;5;132;01m}\u001B[39;00m\u001B[33m\"\u001B[39m\n\u001B[32m    914\u001B[39m     )\n\u001B[32m    916\u001B[39m \u001B[38;5;28;01mif\u001B[39;00m \u001B[38;5;129;01mnot\u001B[39;00m inplace:\n\u001B[32m--> \u001B[39m\u001B[32m917\u001B[39m     tensor = \u001B[43mtensor\u001B[49m\u001B[43m.\u001B[49m\u001B[43mclone\u001B[49m\u001B[43m(\u001B[49m\u001B[43m)\u001B[49m\n\u001B[32m    919\u001B[39m dtype = tensor.dtype\n\u001B[32m    920\u001B[39m mean = torch.as_tensor(mean, dtype=dtype, device=tensor.device)\n",
      "\u001B[31mKeyboardInterrupt\u001B[39m: "
     ]
    }
   ],
   "execution_count": 7
  },
  {
   "metadata": {},
   "cell_type": "markdown",
   "source": [
    "# Grad-CAM evaluation and visualization\n",
    "## Load best model"
   ],
   "id": "a8086a9dfa2a6c50"
  },
  {
   "metadata": {
    "ExecuteTime": {
     "end_time": "2026-03-27T12:55:07.502132Z",
     "start_time": "2026-03-27T12:55:07.473023Z"
    }
   },
   "cell_type": "code",
   "source": [
    "model = SmallCNN(num_classes=num_classes).to(device)\n",
    "model.load_state_dict(torch.load(\"best_smallcnn.pt\", map_location=device))\n",
    "model.eval()\n",
    "\n"
   ],
   "id": "5830febd0ac51bfd",
   "outputs": [
    {
     "data": {
      "text/plain": [
       "SmallCNN(\n",
       "  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))\n",
       "  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))\n",
       "  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))\n",
       "  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)\n",
       "  (gap): AdaptiveAvgPool2d(output_size=(1, 1))\n",
       "  (fc): Linear(in_features=64, out_features=10, bias=True)\n",
       ")"
      ]
     },
     "execution_count": 9,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "execution_count": 9
  },
  {
   "metadata": {
    "ExecuteTime": {
     "end_time": "2026-03-27T13:10:33.554372Z",
     "start_time": "2026-03-27T12:55:09.726406Z"
    }
   },
   "cell_type": "code",
   "source": [
    "batch_size = 32\n",
    "nepochs = 10\n",
    "lr = 0.1\n",
    "units = 100\n",
    "\n",
    "\n",
    "\n",
    "optimizer = torch.optim.SGD(params=model.parameters(), lr = lr)\n",
    "cost_train_sgd, cost_valid_sgd, acc_train_sgd, acc_valid_sgd = (\n",
    "    src.cnn_utils.train_eval(model, optimizer, nepochs, batch_size, train_ds, val_ds, device, entity='MSE_DeLearn_SPR26', project='MPW-CNN', run_name='small_CNN', use_wandb=True))"
   ],
   "id": "88343abbb277f531",
   "outputs": [
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "\u001B[34m\u001B[1mwandb\u001B[0m: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/jdemid/.netrc.\n",
      "\u001B[34m\u001B[1mwandb\u001B[0m: Network error (ConnectionError), entering retry loop.\n",
      "\u001B[34m\u001B[1mwandb\u001B[0m: W&B API key is configured. Use \u001B[1m`wandb login --relogin`\u001B[0m to force relogin\n",
      "\u001B[34m\u001B[1mwandb\u001B[0m: Network error (ConnectionError), entering retry loop.\n",
      "\u001B[34m\u001B[1mwandb\u001B[0m: setting up run mpnunrnz\n",
      "\u001B[34m\u001B[1mwandb\u001B[0m: Tracking run with wandb version 0.25.1\n",
      "\u001B[34m\u001B[1mwandb\u001B[0m: Run data is saved locally in \u001B[35m\u001B[1m/Users/jdemid/Projects/PycharmProjects/deep-learning-mpw/01_MPW-CNN/notebooks/wandb/run-20260327_135519-mpnunrnz\u001B[0m\n",
      "\u001B[34m\u001B[1mwandb\u001B[0m: Run \u001B[1m`wandb offline`\u001B[0m to turn off syncing.\n",
      "\u001B[34m\u001B[1mwandb\u001B[0m: Syncing run \u001B[33msmall_CNN\u001B[0m\n",
      "\u001B[34m\u001B[1mwandb\u001B[0m: ⭐️ View project at \u001B[34m\u001B[4mhttps://wandb.ai/MSE_DeLearn_SPR26/MPW-CNN\u001B[0m\n",
      "\u001B[34m\u001B[1mwandb\u001B[0m: 🚀 View run at \u001B[34m\u001B[4mhttps://wandb.ai/MSE_DeLearn_SPR26/MPW-CNN/runs/mpnunrnz\u001B[0m\n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Epoch 0: nan, 0.101100, nan, 0.100000\n",
      "Epoch 1: nan, 0.100500, nan, 0.100000\n",
      "Epoch 2: nan, 0.100500, nan, 0.100000\n",
      "Epoch 3: nan, 0.100500, nan, 0.100000\n",
      "Epoch 4: nan, 0.100500, nan, 0.100000\n",
      "Epoch 5: nan, 0.100500, nan, 0.100000\n",
      "Epoch 6: nan, 0.100500, nan, 0.100000\n",
      "Epoch 7: nan, 0.100500, nan, 0.100000\n",
      "Epoch 8: nan, 0.100500, nan, 0.100000\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "\u001B[34m\u001B[1mwandb\u001B[0m: updating run metadata\n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Epoch 9: nan, 0.100500, nan, 0.100000\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "\u001B[34m\u001B[1mwandb\u001B[0m: uploading config.yaml; uploading wandb-summary.json\n",
      "\u001B[34m\u001B[1mwandb\u001B[0m: uploading wandb-summary.json\n",
      "\u001B[34m\u001B[1mwandb\u001B[0m: uploading history steps 9-9, summary\n",
      "\u001B[34m\u001B[1mwandb\u001B[0m: \n",
      "\u001B[34m\u001B[1mwandb\u001B[0m: Run history:\n",
      "\u001B[34m\u001B[1mwandb\u001B[0m:          epoch ▁▂▃▃▄▅▆▆▇█\n",
      "\u001B[34m\u001B[1mwandb\u001B[0m:             lr ▁▁▁▁▁▁▁▁▁▁\n",
      "\u001B[34m\u001B[1mwandb\u001B[0m: train_accuracy █▁▁▁▁▁▁▁▁▁\n",
      "\u001B[34m\u001B[1mwandb\u001B[0m:   val_accuracy ▁▁▁▁▁▁▁▁▁▁\n",
      "\u001B[34m\u001B[1mwandb\u001B[0m:             +2 ...\n",
      "\u001B[34m\u001B[1mwandb\u001B[0m: \n",
      "\u001B[34m\u001B[1mwandb\u001B[0m: Run summary:\n",
      "\u001B[34m\u001B[1mwandb\u001B[0m:          epoch 10\n",
      "\u001B[34m\u001B[1mwandb\u001B[0m:             lr 0.1\n",
      "\u001B[34m\u001B[1mwandb\u001B[0m: train_accuracy 0.1005\n",
      "\u001B[34m\u001B[1mwandb\u001B[0m:     train_loss nan\n",
      "\u001B[34m\u001B[1mwandb\u001B[0m:   val_accuracy 0.1\n",
      "\u001B[34m\u001B[1mwandb\u001B[0m:       val_loss nan\n",
      "\u001B[34m\u001B[1mwandb\u001B[0m: \n",
      "\u001B[34m\u001B[1mwandb\u001B[0m: 🚀 View run \u001B[33msmall_CNN\u001B[0m at: \u001B[34m\u001B[4mhttps://wandb.ai/MSE_DeLearn_SPR26/MPW-CNN/runs/mpnunrnz\u001B[0m\n",
      "\u001B[34m\u001B[1mwandb\u001B[0m: ⭐️ View project at: \u001B[34m\u001B[4mhttps://wandb.ai/MSE_DeLearn_SPR26/MPW-CNN\u001B[0m\n",
      "\u001B[34m\u001B[1mwandb\u001B[0m: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)\n",
      "\u001B[34m\u001B[1mwandb\u001B[0m: Find logs at: \u001B[35m\u001B[1m./wandb/run-20260327_135519-mpnunrnz/logs\u001B[0m\n"
     ]
    }
   ],
   "execution_count": 10
  },
  {
   "metadata": {
    "ExecuteTime": {
     "end_time": "2026-03-25T10:45:55.037724Z",
     "start_time": "2026-03-25T10:45:55.008172Z"
    }
   },
   "cell_type": "code",
   "source": [
    "# import cv2\n",
    "# import numpy as np\n",
    "# from pytorch_grad_cam.utils.image import preprocess_image\n",
    "#\n",
    "# file_name = \"0d3211c08b0095d66b7dedaaaa451ab5.jpg\"\n",
    "#\n",
    "# rgb_img = cv2.imread(\"../data/icosimal_img_class_03/validate/cat/\" + file_name)[:, :, ::-1]\n",
    "# rgb_img = cv2.resize(rgb_img, (img_size, img_size))\n",
    "# rgb_img = np.float32(rgb_img) / 255.0\n",
    "#\n",
    "# input_tensor = preprocess_image(\n",
    "#     rgb_img,\n",
    "#     mean=[0.5, 0.5, 0.5],\n",
    "#     std=[0.5, 0.5, 0.5]\n",
    "# ).to(device)"
   ],
   "id": "e11154fcededc449",
   "outputs": [],
   "execution_count": 12
  },
  {
   "metadata": {},
   "cell_type": "markdown",
   "source": "## Inference\n",
   "id": "48390f4f901ec0c0"
  },
  {
   "metadata": {
    "ExecuteTime": {
     "end_time": "2026-03-25T10:45:57.224235Z",
     "start_time": "2026-03-25T10:45:57.211315Z"
    }
   },
   "cell_type": "code",
   "source": [
    "# with torch.no_grad():\n",
    "#     logits = model(input_tensor)\n",
    "#     pred_class = int(logits.argmax(dim=1).item())\n",
    "#     confidence = float(torch.softmax(logits, dim=1)[0, pred_class].item())\n",
    "#\n",
    "# print(\"Predicted: \", class_names[pred_class], \" with confidence: \", confidence)"
   ],
   "id": "6fe9b1af68251886",
   "outputs": [],
   "execution_count": 13
  },
  {
   "metadata": {},
   "cell_type": "markdown",
   "source": "## Grad-CAM",
   "id": "c63244ef614abc6"
  },
  {
   "metadata": {
    "ExecuteTime": {
     "end_time": "2026-03-27T13:10:38.081582Z",
     "start_time": "2026-03-27T13:10:33.556408Z"
    }
   },
   "cell_type": "code",
   "source": [
    "import os\n",
    "from pathlib import Path\n",
    "\n",
    "import cv2\n",
    "import numpy as np\n",
    "import torch\n",
    "\n",
    "from pytorch_grad_cam import GradCAM\n",
    "from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget\n",
    "from pytorch_grad_cam.utils.image import preprocess_image, show_cam_on_image\n",
    "\n",
    "\n",
    "def inspect_image_with_gradcam_from_folder_label(\n",
    "    model,\n",
    "    image_path,\n",
    "    target_layers,\n",
    "    class_names,\n",
    "    class_to_idx,\n",
    "    mean,\n",
    "    std,\n",
    "    device=None,\n",
    "    resize_to=None,\n",
    "    output_dir=\"gradcam_outputs\",\n",
    "    eigen_smooth=False,\n",
    "    aug_smooth=False,\n",
    "):\n",
    "    \"\"\"\n",
    "    Assumes image_path looks like:\n",
    "        .../<split>/<class_name>/<image_file>\n",
    "\n",
    "    True class is inferred from the parent folder name.\n",
    "    \"\"\"\n",
    "\n",
    "    if device is None:\n",
    "        device = next(model.parameters()).device\n",
    "\n",
    "    model.eval()\n",
    "    image_path = Path(image_path)\n",
    "    output_dir = Path(output_dir)\n",
    "    output_dir.mkdir(parents=True, exist_ok=True)\n",
    "\n",
    "    # ---- infer true class from parent folder ----\n",
    "    true_class_name = image_path.parent.name\n",
    "    if true_class_name not in class_to_idx:\n",
    "        raise ValueError(\n",
    "            f\"Parent folder '{true_class_name}' is not in class_to_idx: {list(class_to_idx.keys())}\"\n",
    "        )\n",
    "    true_class = class_to_idx[true_class_name]\n",
    "\n",
    "    # ---- read image ----\n",
    "    bgr = cv2.imread(str(image_path))\n",
    "    if bgr is None:\n",
    "        raise FileNotFoundError(f\"Could not read image: {image_path}\")\n",
    "\n",
    "    rgb_img = bgr[:, :, ::-1]\n",
    "    if resize_to is not None:\n",
    "        rgb_img = cv2.resize(rgb_img, resize_to)\n",
    "\n",
    "    rgb_img = np.float32(rgb_img) / 255.0\n",
    "\n",
    "    # ---- preprocess ----\n",
    "    input_tensor = preprocess_image(\n",
    "        rgb_img,\n",
    "        mean=mean,\n",
    "        std=std\n",
    "    ).to(device)\n",
    "\n",
    "    # ---- predict ----\n",
    "    with torch.no_grad():\n",
    "        logits = model(input_tensor)\n",
    "        probs = torch.softmax(logits, dim=1)\n",
    "\n",
    "        pred_class = int(logits.argmax(dim=1).item())\n",
    "        pred_conf = float(probs[0, pred_class].item())\n",
    "        true_conf = float(probs[0, true_class].item())\n",
    "        is_correct = (pred_class == true_class)\n",
    "\n",
    "    result = {\n",
    "        \"image_path\": str(image_path),\n",
    "        \"true_class_idx\": true_class,\n",
    "        \"true_class_name\": class_names[true_class],\n",
    "        \"true_class_confidence\": true_conf,\n",
    "        \"pred_class_idx\": pred_class,\n",
    "        \"pred_class_name\": class_names[pred_class],\n",
    "        \"pred_confidence\": pred_conf,\n",
    "        \"is_correct\": is_correct,\n",
    "        \"saved_files\": [],\n",
    "    }\n",
    "\n",
    "    stem = image_path.stem\n",
    "\n",
    "    with GradCAM(model=model, target_layers=target_layers) as cam:\n",
    "        if is_correct:\n",
    "            # one heatmap only\n",
    "            grayscale_cam = cam(\n",
    "                input_tensor=input_tensor,\n",
    "                targets=[ClassifierOutputTarget(pred_class)],\n",
    "                eigen_smooth=eigen_smooth,\n",
    "                aug_smooth=aug_smooth,\n",
    "            )[0, :]\n",
    "\n",
    "            overlay = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)\n",
    "            out_path = output_dir / f\"{stem}__correct_{class_names[pred_class]}.jpg\"\n",
    "            cv2.imwrite(str(out_path), overlay[:, :, ::-1])\n",
    "            result[\"saved_files\"].append(str(out_path))\n",
    "\n",
    "        else:\n",
    "            # predicted-class heatmap\n",
    "            cam_pred = cam(\n",
    "                input_tensor=input_tensor,\n",
    "                targets=[ClassifierOutputTarget(pred_class)],\n",
    "                eigen_smooth=eigen_smooth,\n",
    "                aug_smooth=aug_smooth,\n",
    "            )[0, :]\n",
    "\n",
    "            overlay_pred = show_cam_on_image(rgb_img, cam_pred, use_rgb=True)\n",
    "            out_pred = output_dir / f\"{stem}__pred_{class_names[pred_class]}.jpg\"\n",
    "            cv2.imwrite(str(out_pred), overlay_pred[:, :, ::-1])\n",
    "            result[\"saved_files\"].append(str(out_pred))\n",
    "\n",
    "            # true-class heatmap\n",
    "            cam_true = cam(\n",
    "                input_tensor=input_tensor,\n",
    "                targets=[ClassifierOutputTarget(true_class)],\n",
    "                eigen_smooth=eigen_smooth,\n",
    "                aug_smooth=aug_smooth,\n",
    "            )[0, :]\n",
    "\n",
    "            overlay_true = show_cam_on_image(rgb_img, cam_true, use_rgb=True)\n",
    "            out_true = output_dir / f\"{stem}__true_{class_names[true_class]}.jpg\"\n",
    "            cv2.imwrite(str(out_true), overlay_true[:, :, ::-1])\n",
    "            result[\"saved_files\"].append(str(out_true))\n",
    "\n",
    "    return result"
   ],
   "id": "8c59b233c3039fad",
   "outputs": [],
   "execution_count": 11
  },
  {
   "metadata": {
    "ExecuteTime": {
     "end_time": "2026-03-27T13:10:38.146736Z",
     "start_time": "2026-03-27T13:10:38.085775Z"
    }
   },
   "cell_type": "code",
   "source": [
    "img_path = os.path.join(MPW_CNN_DIR, \"data\", \"icosimal_img_class_03\", \"validate\", \"cat\", \"1a63f354a05e5ce823d7b0308394c451.jpg\")\n",
    "\n",
    "result = inspect_image_with_gradcam_from_folder_label(\n",
    "    model=model,\n",
    "    image_path=img_path,\n",
    "    target_layers=[model.conv3],   # replace if your last conv layer is different\n",
    "    class_names=val_ds.classes,\n",
    "    class_to_idx=val_ds.class_to_idx,\n",
    "    mean=[0.5, 0.5, 0.5],\n",
    "    std=[0.5, 0.5, 0.5],\n",
    "    device=device,\n",
    "    resize_to=(128, 128),\n",
    "    output_dir=\"gradcam_outputs\",\n",
    ")\n",
    "\n",
    "print(\"predicted :\", result[\"pred_class_name\"], result[\"pred_confidence\"])\n",
    "print(\"true      :\", result[\"true_class_name\"], result[\"true_class_confidence\"])\n",
    "print(\"correct   :\", result[\"is_correct\"])\n",
    "print(\"files     :\", result[\"saved_files\"])"
   ],
   "id": "46634840985218a9",
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "predicted : cat nan\n",
      "true      : cat nan\n",
      "correct   : True\n",
      "files     : ['gradcam_outputs/1a63f354a05e5ce823d7b0308394c451__correct_cat.jpg']\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "/Users/jdemid/Projects/PycharmProjects/deep-learning-mpw/.venv/lib/python3.12/site-packages/pytorch_grad_cam/utils/image.py:50: RuntimeWarning: invalid value encountered in cast\n",
      "  heatmap = cv2.applyColorMap(np.uint8(255 * mask), colormap)\n"
     ]
    }
   ],
   "execution_count": 12
  },
  {
   "metadata": {},
   "cell_type": "markdown",
   "source": "## Model metrics and confusion matrix",
   "id": "30124edf8188d68d"
  },
  {
   "cell_type": "code",
   "id": "initial_id",
   "metadata": {
    "collapsed": true,
    "ExecuteTime": {
     "end_time": "2026-03-25T12:08:26.773843Z",
     "start_time": "2026-03-25T12:08:21.771825Z"
    }
   },
   "source": [
    "\n",
    "from sklearn.metrics import (\n",
    "    accuracy_score,\n",
    "    confusion_matrix,\n",
    "    classification_report,\n",
    "    ConfusionMatrixDisplay,\n",
    ")\n",
    "\n",
    "# expects:\n",
    "# - model already loaded and in eval() mode\n",
    "# - val_loader\n",
    "# - class_names (e.g. train_ds.classes)\n",
    "# - device\n",
    "\n",
    "@torch.no_grad()\n",
    "def evaluate_on_loader(model, loader, class_names, device):\n",
    "    model.eval()\n",
    "\n",
    "    all_true = []\n",
    "    all_pred = []\n",
    "    all_probs = []\n",
    "\n",
    "    for images, labels in loader:\n",
    "        images = images.to(device)\n",
    "        labels = labels.to(device)\n",
    "\n",
    "        logits = model(images)\n",
    "        probs = torch.softmax(logits, dim=1)\n",
    "        preds = probs.argmax(dim=1)\n",
    "\n",
    "        all_true.extend(labels.cpu().numpy())\n",
    "        all_pred.extend(preds.cpu().numpy())\n",
    "        all_probs.extend(probs.cpu().numpy())\n",
    "\n",
    "    y_true = np.array(all_true)\n",
    "    y_pred = np.array(all_pred)\n",
    "    y_prob = np.array(all_probs)\n",
    "\n",
    "    # overall accuracy\n",
    "    acc = accuracy_score(y_true, y_pred)\n",
    "    print(f\"Validation accuracy: {acc:.4f}\")\n",
    "\n",
    "    # text metrics\n",
    "    print(\"\\nClassification report:\")\n",
    "    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))\n",
    "\n",
    "    # confusion matrices\n",
    "    cm = confusion_matrix(y_true, y_pred)\n",
    "    cm_norm = confusion_matrix(y_true, y_pred, normalize=\"true\")\n",
    "\n",
    "    # per-class accuracy = recall per class = diag / row sum\n",
    "    per_class_acc = np.divide(\n",
    "        np.diag(cm),\n",
    "        cm.sum(axis=1),\n",
    "        out=np.zeros(len(class_names), dtype=float),\n",
    "        where=cm.sum(axis=1) != 0\n",
    "    )\n",
    "\n",
    "    # confidence of predicted class\n",
    "    pred_conf = y_prob[np.arange(len(y_pred)), y_pred]\n",
    "    correct_mask = (y_true == y_pred)\n",
    "\n",
    "    # ---- plots ----\n",
    "\n",
    "    # 1) per-class accuracy bar chart\n",
    "    plt.figure(figsize=(10, 4))\n",
    "    plt.bar(class_names, per_class_acc)\n",
    "    plt.ylim(0, 1)\n",
    "    plt.ylabel(\"Per-class accuracy\")\n",
    "    plt.title(f\"Validation accuracy = {acc:.4f}\")\n",
    "    plt.xticks(rotation=45, ha=\"right\")\n",
    "    plt.grid(axis=\"y\")\n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "\n",
    "    # 2) raw confusion matrix\n",
    "    fig, ax = plt.subplots(figsize=(7, 7))\n",
    "    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)\n",
    "    disp.plot(ax=ax, xticks_rotation=45, colorbar=False, cmap=\"Blues\")\n",
    "    ax.set_title(\"Confusion Matrix (Counts)\")\n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "\n",
    "    # 3) normalized confusion matrix\n",
    "    fig, ax = plt.subplots(figsize=(7, 7))\n",
    "    disp = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=class_names)\n",
    "    disp.plot(ax=ax, xticks_rotation=45, values_format=\".2f\", colorbar=False, cmap=\"Blues\")\n",
    "    ax.set_title(\"Confusion Matrix (Row-normalized)\")\n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "\n",
    "    # 4) confidence histogram: correct vs wrong\n",
    "    plt.figure(figsize=(8, 4))\n",
    "    plt.hist(pred_conf[correct_mask], bins=20, alpha=0.7, label=\"Correct\")\n",
    "    plt.hist(pred_conf[~correct_mask], bins=20, alpha=0.7, label=\"Wrong\")\n",
    "    plt.xlabel(\"Predicted-class confidence\")\n",
    "    plt.ylabel(\"Count\")\n",
    "    plt.title(\"Confidence Distribution\")\n",
    "    plt.legend()\n",
    "    plt.tight_layout()\n",
    "    plt.grid(axis=\"y\")\n",
    "    plt.show()\n",
    "\n",
    "    # top-N most confident mistakes\n",
    "    N = 10\n",
    "    pred_conf = y_prob[np.arange(len(y_pred)), y_pred]\n",
    "    wrong_idx = np.where(y_true != y_pred)[0]\n",
    "    top_wrong = wrong_idx[np.argsort(pred_conf[wrong_idx])[::-1][:N]]\n",
    "\n",
    "    print(\"Most confident mistakes:\")\n",
    "    for i in top_wrong:\n",
    "        print(\n",
    "            f\"idx={i:4d}  true={class_names[y_true[i]]:>15s}  \"\n",
    "            f\"pred={class_names[y_pred[i]]:>15s}  conf={pred_conf[i]:.4f}\"\n",
    "        )\n",
    "\n",
    "    return {\n",
    "        \"accuracy\": acc,\n",
    "        \"y_true\": y_true,\n",
    "        \"y_pred\": y_pred,\n",
    "        \"y_prob\": y_prob,\n",
    "        \"cm\": cm,\n",
    "        \"cm_norm\": cm_norm,\n",
    "        \"per_class_acc\": per_class_acc,\n",
    "    }\n",
    "\n",
    "# run it\n",
    "results = evaluate_on_loader(model, val_loader, class_names, device)"
   ],
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Validation accuracy: 0.4515\n",
      "\n",
      "Classification report:\n",
      "              precision    recall  f1-score   support\n",
      "\n",
      "         cat     0.3740    0.4700    0.4165       600\n",
      "     chicken     0.4350    0.5633    0.4909       600\n",
      "         cow     0.4093    0.3083    0.3517       600\n",
      "         dog     0.3023    0.1733    0.2203       600\n",
      "    elephant     0.4460    0.5850    0.5061       600\n",
      "       horse     0.4511    0.3000    0.3604       600\n",
      "      rabbit     0.4163    0.3233    0.3640       600\n",
      "       sheep     0.3610    0.5650    0.4405       600\n",
      "    squirrel     0.4340    0.3617    0.3945       600\n",
      "       zebra     0.8918    0.8650    0.8782       600\n",
      "\n",
      "    accuracy                         0.4515      6000\n",
      "   macro avg     0.4521    0.4515    0.4423      6000\n",
      "weighted avg     0.4521    0.4515    0.4423      6000\n",
      "\n"
     ]
    },
    {
     "data": {
      "text/plain": [
       "<Figure size 1000x400 with 1 Axes>"
      ],
      "image/png": "..."
     },
     "metadata": {},
     "output_type": "display_data",
     "jetTransient": {
      "display_id": null
     }
    },
    {
     "data": {
      "text/plain": [
       "<Figure size 640x480 with 0 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data",
     "jetTransient": {
      "display_id": null
     }
    },
    {
     "data": {
      "text/plain": [
       "<Figure size 700x700 with 1 Axes>"
      ],
      "image/png": "..."
     },
     "metadata": {},
     "output_type": "display_data",
     "jetTransient": {
      "display_id": null
     }
    },
    {
     "data": {
      "text/plain": [
       "<Figure size 700x700 with 1 Axes>"
      ],
      "image/png": "..."
     },
     "metadata": {},
     "output_type": "display_data",
     "jetTransient": {
      "display_id": null
     }
    },
    {
     "data": {
      "text/plain": [
       "<Figure size 800x400 with 1 Axes>"
      ],
      "image/png": "..."
     },
     "metadata": {},
     "output_type": "display_data",
     "jetTransient": {
      "display_id": null
     }
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Most confident mistakes:\n",
      "idx=1873  true=            dog  pred=          zebra  conf=1.0000\n",
      "idx=4770  true=          sheep  pred=          zebra  conf=1.0000\n",
      "idx=  66  true=            cat  pred=          zebra  conf=1.0000\n",
      "idx=3733  true=         rabbit  pred=          zebra  conf=0.9999\n",
      "idx=4764  true=          sheep  pred=          zebra  conf=0.9999\n",
      "idx= 290  true=            cat  pred=        chicken  conf=0.9999\n",
      "idx=2921  true=       elephant  pred=          zebra  conf=0.9992\n",
      "idx=3986  true=         rabbit  pred=        chicken  conf=0.9946\n",
      "idx=1173  true=        chicken  pred=          zebra  conf=0.9856\n",
      "idx=5683  true=          zebra  pred=        chicken  conf=0.9840\n"
     ]
    }
   ],
   "execution_count": 11
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 2
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython2",
   "version": "2.7.6"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}
